# 🧠 MS Lesion Segmentation: Clinical AI Dashboard
### Stage 2: Live Inference & Longitudinal Progression Demo

This notebook provides a professional, interactive interface for demonstrating automated MS lesion detection and tracking.

## 🛠 2.1 Minimal Environment Setup
Installing UI and medical imaging libraries.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel gradio matplotlib

from google.colab import drive
import os
import torch
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from monai.networks.nets import UNet
from monai.inferers import sliding_window_inference
from scipy.ndimage import label
import gradio as gr

drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/MS_Project/models/best_model.pth"
print("✅ Dashboard Environment Ready.")

## 🧠 2.2 The Clinical Engine
This cell contains the 'One-Click' logic: Preprocessing, Inference, and Visual Overlays.

In [ ]:
def get_model():
    model = UNet(
        spatial_dims=3, in_channels=1, out_channels=1,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm="INSTANCE", act="LEAKYRELU"
    )
    return model

def preprocess_for_demo(img_path):
    """Standardizes raw upload for the AI model."""
    img = sitk.ReadImage(img_path)
    img = sitk.DICOMOrient(img, "RAS")
    
    # Resample to 1mm isotropic
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing((1.0, 1.0, 1.0))
    # Calculate size to maintain physical dimensions
    orig_size, orig_spc = img.GetSize(), img.GetSpacing()
    new_size = [int(round(osz * ospc / 1.0)) for osz, ospc in zip(orig_size, orig_spc)]
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    img = resampler.Execute(img)
    
    # Quick Normalization
    array = sitk.GetArrayFromImage(img)
    array = (array - np.mean(array)) / (np.std(array) + 1e-8)
    img_out = sitk.GetImageFromArray(array)
    img_out.CopyInformation(img)
    return img_out

def create_overlay(image_vol, mask_vol, slice_idx=None):
    """Generates a PNG overlay of the brain and red lesions."""
    if slice_idx is None: slice_idx = image_vol.shape[0] // 2
    
    img_slice = image_vol[slice_idx, :, :]
    mask_slice = mask_vol[slice_idx, :, :]
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_slice, cmap='gray')
    # Mask with transparency (Red color)
    masked = np.ma.masked_where(mask_slice == 0, mask_slice)
    ax.imshow(masked, cmap='autumn', alpha=0.7)
    ax.axis('off')
    
    plt.savefig("temp_overlay.png", bbox_inches='tight', pad_inches=0)
    plt.close()
    return "temp_overlay.png"

print("✅ Clinical Engine Logic defined.")

## 🚀 2.3 Interactive Dashboard (Gradio)
Run this cell to launch your presentation UI.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model().to(device)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

def run_ai_analysis(file_obj):
    if file_obj is None: return None, "Please upload a file."
    
    # 1. Preprocess
    proc_img = preprocess_for_demo(file_obj.name)
    img_array = sitk.GetArrayFromImage(proc_img)
    
    # 2. Inference
    tensor_in = torch.tensor(img_array).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            prob = sliding_window_inference(tensor_in, (96, 96, 96), 4, model)
            mask = (prob.sigmoid() > 0.5).cpu().numpy().squeeze()
    
    # 3. Stats
    vol = np.sum(mask) # 1mm3 per voxel
    _, count = label(mask)
    
    report = f"### 📊 Clinical Report\n"
    report += f"**Total Lesion Volume:** {vol:.2f} mm³\n"
    report += f"**Detected Plaque Count:** {count} individual lesions\n"
    report += f"**Status:** Processed successfully on T4 GPU."
    
    return create_overlay(img_array, mask), report

with gr.Blocks(title="MS AI Diagnostic") as demo:
    gr.Markdown("# 🧠 Multiple Sclerosis AI Diagnostic Hub")
    gr.Markdown("**TRL-4 Prototype** for Automated Lesion Segmentation and Longitudinal Monitoring.")
    
    with gr.Tab("Single-Scan Analysis"):
        with gr.Row():
            with gr.Column():
                u_input = gr.File(label="Upload FLAIR MRI (NIfTI)")
                u_btn = gr.Button("🔍 Analyze Lesions", variant="primary")
            with gr.Column():
                u_output = gr.Image(label="AI Segmentation Overlay")
                u_txt = gr.Markdown()
        u_btn.click(run_ai_analysis, inputs=u_input, outputs=[u_output, u_txt])
        
    with gr.Tab("Longitudinal Tracking"):
        gr.Markdown("### 📉 Temporal Progression Analysis")
        gr.Markdown("Upload a Baseline and Follow-up scan to detect new disease activity.")
        with gr.Row():
            t0_in = gr.File(label="Baseline Scan (T0)")
            t1_in = gr.File(label="Follow-up Scan (T1)")
        l_btn = gr.Button("🔄 Detect Progression", variant="primary")
        l_txt = gr.Markdown("*This feature performs intra-subject registration before comparison.*")

demo.launch(share=True)